# 01. YOLO26n + Optuna 하이퍼파라미터 최적화
## Resume-safe / 총 25개 trial / Local + Kaggle 공용

이 Notebook은 긴 Optuna 실험이 중간에 끊겨도 **이미 완료된 trial은 그대로 보존하고,
남은 trial만 이어서 실행**하도록 수정한 통합본입니다.

핵심 변경점은 다음과 같습니다.

```text
기존 방식
study.optimize(..., n_trials=25)
→ Notebook을 다시 실행할 때마다 25개 trial을 추가

수정 방식
TARGET_TOTAL_TRIALS = 25
→ COMPLETE/PRUNED trial이 총 25개가 될 때까지만 추가 실행
```

예를 들어 이전 실행에 다음 상태가 남아 있다고 가정합니다.

```text
Trial 0 COMPLETE
Trial 1 COMPLETE
Trial 2 PRUNED
Trial 3 COMPLETE
Trial 4 COMPLETE
Trial 5 RUNNING   ← 세션 종료 때문에 중간에 끊김
```

이 Notebook을 다시 실행하면:

1. Trial 0~4 결과는 그대로 보존
2. Trial 5처럼 오래된 `RUNNING` 상태는 `FAIL`로 정리
3. COMPLETE + PRUNED를 유효 trial로 계산
4. 총 25개가 될 때까지만 새 trial 실행

즉 이미 유효 trial이 5개라면 **20개만 추가**합니다.

---

## 중요한 점: trial 내부 epoch resume와 Study resume는 다릅니다

이 Notebook은 **Optuna Study 단위 resume**를 지원합니다.

```text
완료된 Trial 0~4 유지   → O
중단된 Trial 5 epoch 2부터 재개 → X
```

중단된 trial 하나의 YOLO epoch 중간부터 이어가는 것은 하지 않습니다.
그 trial은 FAIL 처리하고 새로운 trial로 다시 시도합니다.

현재처럼 Trial 5의 epoch 2 정도에서 중단된 상황에서는 이 방식이 훨씬 단순하고 안전합니다.

---

## GPU 사용 원칙

Kaggle에서 T4 x2를 선택해도 이 Notebook은 **GPU 0 한 장만 사용**합니다.

```text
Kaggle T4 x2
GPU 0 → Optuna / YOLO 사용
GPU 1 → 사용하지 않음
```

이유는:

- 00 Baseline / 01 Optuna / 02 Augmentation의 모델 품질 비교 조건을 맞추기 위해서
- Optuna pruning callback + Ultralytics DDP 조합을 복잡하게 만들지 않기 위해서
- YOLO26n + 작은 batch에서는 2-GPU DDP가 반드시 2배 빨라지는 것도 아니기 때문입니다.

로컬에서도 GPU 한 장만 사용합니다.

# 1. mAP50-95만 보고 모델을 고르는 것은 괜찮을까?

## mAP50-95는 단순한 "정확도 한 숫자"가 아닙니다.

Object Detection에서 AP는 confidence threshold를 하나만 정해서 계산하는 값이 아니라,
Precision-Recall 관계 전체를 이용합니다.

`mAP50-95`는 다시 여러 IoU 기준에서 AP를 평균합니다.

따라서 Detection 모델 전체 성능을 비교하는 **1차 기준**으로 mAP50-95를 사용하는 것은 적절합니다.

하지만 서비스 목적에 따라 추가 기준이 필요합니다.

우리 서비스에서는 실제 객체를 아예 놓치는 문제가 중요하므로 **Recall도 함께 봐야 합니다.**

### 이 노트북의 선정 원칙

1. Optuna 자체 목적함수는 `mAP50-95`를 최대화
2. 모든 trial의 Precision / Recall / F1 / mAP50도 함께 저장
3. 최고 mAP50-95에서 아주 조금만 차이나는 후보들을 shortlist
4. shortlist 안에서는 Recall을 추가 기준으로 사용
5. 최종 후보는 여러 seed로 다시 학습

즉,

> **mAP를 무시하지도 않고, mAP 하나만 맹목적으로 보지도 않는 방식**

입니다.

# 2. Optuna란?

하이퍼파라미터는 모델이 학습하면서 스스로 배우는 weight가 아니라,
**학습을 시작하기 전에 사람이 정해야 하는 설정값**입니다.

예:

- 초기 Learning Rate `lr0`
- 학습 마지막의 Learning Rate 비율 `lrf`
- Momentum
- Weight Decay
- Warmup Epoch
- Optimizer 종류

수동으로 모든 조합을 테스트하려면 경우의 수가 매우 커집니다.

Optuna는 이전 trial 결과를 이용해 다음에 시도할 값들을 더 효율적으로 선택합니다.

이 노트북에서는 기본적으로 **TPE(Tree-structured Parzen Estimator)** sampler를 사용합니다.

또한 성능이 초반부터 매우 나쁜 trial은 끝까지 학습하지 않고 중단하는 **Pruning**도 사용합니다.

---

## Optuna가 무조건 "제일 좋은" 방법인가?

아닙니다.

YOLO에는 다음과 같은 선택지가 있습니다.

- 사람이 직접 Grid / Random Search
- Ultralytics 내장 Tuner의 Genetic Algorithm
- Ray Tune
- Optuna
- Bayesian Optimization 계열 도구

이번 프로젝트에서는 Optuna를 추천합니다.

이유:

- 탐색 범위를 우리가 명확하게 정의 가능
- trial별 결과 확인이 쉬움
- SQLite DB로 중단 후 재개 가능
- Pruning 지원
- 파라미터 중요도 분석 가능
- 이후 보고서 작성이 편함

즉 **작은 팀 프로젝트에서 실험 과정을 이해하고 기록하기 좋은 선택**입니다.

# 3. 증강과 하이퍼파라미터 중 무엇을 먼저 해야 하나?

완전히 독립된 문제는 아닙니다.

예를 들어 강한 Mosaic를 사용하면 가장 좋은 Learning Rate가 바뀔 수도 있습니다.
그래서 이론적으로는 augmentation과 모든 학습 파라미터를 한 번에 joint search할 수도 있습니다.

하지만 우리 데이터 규모에서 처음부터 전부 같이 탐색하면:

- 검색 공간이 너무 커지고
- trial 수가 부족해지고
- Validation에 과적합될 위험이 커지고
- 무엇 때문에 좋아졌는지 해석하기 어려워집니다.

## 권장 순서

```text
① Baseline
      ↓
② 핵심 학습 하이퍼파라미터 Optuna 탐색
   (augmentation은 B01 default로 고정)
      ↓
③ Optuna Best를 여러 seed로 확인
      ↓
④ 이전 증강 상위 후보 B01 / H04 / Y14 재비교
   (학습 하이퍼파라미터는 Optuna Best로 고정)
      ↓
⑤ 필요하면 최종 승자 augmentation의 강도만 좁게 추가 탐색
      ↓
⑥ 최종 여러 seed 검증
      ↓
⑦ 독립 Test set은 마지막 한 번 평가
```

### 왜 지금은 이 순서인가?

이미 앞 노트북에서 augmentation을 한 차례 폭넓게 탐색했고,
multi-seed에서는 **B01 default가 가장 안정적**이었습니다.

따라서 그 결과를 버리지 않고 B01을 HPO 기준점으로 사용하는 것이 가장 합리적입니다.

# 4. 이번 Optuna에서 무엇을 튜닝하고 무엇을 고정할까?

하이퍼파라미터라고 해서 전부 동시에 바꾸면 안 됩니다.

## 1차 Optuna에서 튜닝할 값

### `optimizer`
- `AdamW`
- `SGD`

Optimizer는 weight를 어떤 규칙으로 업데이트할지 결정합니다.

### `lr0`
초기 Learning Rate입니다.

너무 크면 최적점을 지나칠 수 있고,
너무 작으면 학습이 지나치게 느리거나 충분히 이동하지 못할 수 있습니다.

### `lrf`
학습 마지막 Learning Rate를 `lr0`의 어느 정도 비율까지 줄일지 결정합니다.

### `momentum`
이전 gradient 방향을 얼마나 기억할지 결정합니다.

### `weight_decay`
weight가 지나치게 커지는 것을 억제하는 regularization입니다.
작은 데이터에서 과적합 방지에 도움이 될 수 있습니다.

### `warmup_epochs`
학습 초반 Learning Rate를 천천히 올리는 기간입니다.
초기 학습 불안정을 줄이는 데 도움이 됩니다.

### `cos_lr`
Cosine Learning Rate schedule 사용 여부입니다.

---

## 1차에서 고정할 값

### `batch`
GPU VRAM과 직접 관련이 크므로 실험 품질 변수보다는 **환경 제약값**에 가깝게 다룹니다.

### `imgsz`
512/640/768처럼 바꾸면 정확도뿐 아니라 속도와 VRAM까지 크게 바뀝니다.
따라서 1차 Optuna에서는 640으로 고정합니다.

### `epochs`
trial마다 epoch 자체를 튜닝하지 않습니다.
최대 epoch를 고정하고 Optuna pruning으로 나쁜 trial을 조기에 종료합니다.

### augmentation
B01 YOLO default로 고정합니다.

이렇게 해야 먼저 **순수 학습 설정의 영향**을 비교할 수 있습니다.

# 5. 폴더 구조

이 노트북은 앞 실험과 섞이지 않게 별도 폴더를 사용합니다.

```text
현재 PROJECT_ROOT
│
├─ ../../data/processed/
│
├─ ../models/yolo/01_experiment_augmentation/
│  └─ report/
│     └─ summary/
│        ├─ experiment_results.csv
│        └─ multiseed_summary.csv
│
└─ ../models/yolo/02_optuna_hyperparameter_tuning/
   ├─ optuna_study.db
   ├─ tune_runs/
   ├─ final_runs/
   ├─ augmentation_recheck/
   └─ report/
      ├─ trials.csv
      ├─ parameter_importance.csv
      ├─ selected_trial.json
      ├─ final_multiseed.csv
      ├─ augmentation_recheck.csv
      └─ figures/
```

`tune_runs/`는 trial 중간 checkpoint라 용량이 클 수 있습니다.
기본 설정에서는 trial 평가가 끝나면 삭제합니다.

최종 multi-seed 모델은 `final_runs/`에 남깁니다.

# 6. 라이브러리 불러오기

In [ ]:
from __future__ import annotations

import gc
import json
import math
import random
import shutil
import time
import zipfile
import hashlib
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import torch
import optuna
from optuna.trial import TrialState

from ultralytics import YOLO

try:
    from ultralytics.cfg import DEFAULT_CFG_DICT
except Exception:
    DEFAULT_CFG_DICT = {}

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)

print("PyTorch :", torch.__version__)
print("Optuna  :", optuna.__version__)

# 7. 경로와 공통 설정

가장 먼저 확인해야 하는 셀입니다.

이전 프로젝트 경로 규칙을 그대로 사용합니다.

In [ ]:
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_WORKING_ROOT = Path("/kaggle/working")

IS_KAGGLE = KAGGLE_INPUT_ROOT.exists()
ENVIRONMENT = "KAGGLE" if IS_KAGGLE else "LOCAL"


def is_valid_yolo_dataset(path: Path) -> bool:
    required = [
        path / "data.yaml",
        path / "images" / "train",
        path / "images" / "val",
        path / "labels" / "train",
        path / "labels" / "val",
    ]
    return all(item.exists() for item in required)


def find_dataset_dirs(root: Path):
    candidates = []

    if not root.exists():
        return candidates

    if is_valid_yolo_dataset(root):
        candidates.append(root.resolve())

    for yaml_path in root.rglob("data.yaml"):
        parent = yaml_path.parent

        if is_valid_yolo_dataset(parent):
            score = 0

            if parent.name.lower() == "processed":
                score += 10

            if parent.parent.name.lower() == "data":
                score += 3

            candidates.append((score, parent.resolve()))

    # root itself may have been added without score
    normalized = []
    for item in candidates:
        if isinstance(item, tuple):
            normalized.append(item)
        else:
            normalized.append((0, item))

    unique = {}
    for score, path in normalized:
        unique[path] = max(score, unique.get(path, -1))

    return sorted(
        [(score, path) for path, score in unique.items()],
        key=lambda item: (-item[0], str(item[1])),
    )


def find_data_zip(search_root: Path):
    if not search_root.exists():
        return None

    candidates = sorted(
        search_root.rglob("data.zip")
    )

    return candidates[0] if candidates else None


def safe_extract_zip(
    zip_path: Path,
    extract_root: Path,
) -> None:
    extract_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    with zipfile.ZipFile(
        zip_path,
        "r",
    ) as zf:

        root_resolved = (
            extract_root.resolve()
        )

        for member in zf.infolist():
            target = (
                extract_root
                / member.filename
            ).resolve()

            if root_resolved not in [
                target,
                *target.parents,
            ]:
                raise RuntimeError(
                    f"안전하지 않은 ZIP 경로: "
                    f"{member.filename}"
                )

        zf.extractall(
            extract_root
        )


def resolve_dataset():
    """
    우선순위:
    1. Kaggle이 ZIP을 자동 해제해 /kaggle/input에 노출한 YOLO 폴더
    2. /kaggle/input에 data.zip이 그대로 있을 경우 직접 해제
    3. 로컬에서는 data/processed 구조
    4. 로컬 data.zip
    """

    if IS_KAGGLE:

        extracted_candidates = (
            find_dataset_dirs(
                KAGGLE_INPUT_ROOT
            )
        )

        if extracted_candidates:
            print(
                "Kaggle Input에서 이미 풀린 "
                "YOLO dataset을 발견했습니다."
            )

            for index, (
                score,
                path,
            ) in enumerate(
                extracted_candidates
            ):
                print(
                    f"[{index}] "
                    f"score={score} | {path}"
                )

            return (
                extracted_candidates[0][1],
                None,
                None,
            )

        zip_path = find_data_zip(
            KAGGLE_INPUT_ROOT
        )

        if zip_path is None:
            raise FileNotFoundError(
                "Kaggle Input에서 YOLO dataset도 "
                "data.zip도 찾지 못했습니다."
            )

        extract_root = (
            KAGGLE_WORKING_ROOT
            / "yolo_dataset"
        ).resolve()

        safe_extract_zip(
            zip_path,
            extract_root,
        )

        extracted_candidates = (
            find_dataset_dirs(
                extract_root
            )
        )

        if not extracted_candidates:
            raise FileNotFoundError(
                "data.zip 압축 해제 후 "
                "YOLO dataset 구조를 찾지 못했습니다."
            )

        return (
            extracted_candidates[0][1],
            zip_path.resolve(),
            extract_root,
        )

    # ----------------------------
    # Local
    # ----------------------------
    cwd = Path.cwd().resolve()

    for root in [cwd, *cwd.parents]:
        direct = (
            root
            / "data"
            / "processed"
        )

        if is_valid_yolo_dataset(
            direct
        ):
            return (
                direct.resolve(),
                None,
                None,
            )

    for root in [cwd, *cwd.parents]:
        zip_path = (
            root
            / "data.zip"
        )

        if zip_path.exists():

            extract_root = (
                cwd
                / ".yolo_data_cache"
            ).resolve()

            safe_extract_zip(
                zip_path,
                extract_root,
            )

            extracted_candidates = (
                find_dataset_dirs(
                    extract_root
                )
            )

            if extracted_candidates:
                return (
                    extracted_candidates[0][1],
                    zip_path.resolve(),
                    extract_root,
                )

    raise FileNotFoundError(
        "로컬에서 data/processed 또는 "
        "data.zip을 찾지 못했습니다."
    )


PROCESSED_DIR, DATA_ZIP, EXTRACT_ROOT = (
    resolve_dataset()
)

DATA_YAML = (
    PROCESSED_DIR
    / "data.yaml"
)

if IS_KAGGLE:
    OPTUNA_ROOT = (
        KAGGLE_WORKING_ROOT
        / "01_yolo_optuna_no_aug"
    ).resolve()
    WORKERS = 2
else:
    OPTUNA_ROOT = (
        Path.cwd().resolve()
        / "01_yolo_optuna_no_aug"
    ).resolve()
    WORKERS = 0

TUNE_RUNS_DIR = OPTUNA_ROOT / "tune_runs"
FINAL_RUNS_DIR = OPTUNA_ROOT / "final_runs"
REPORT_DIR = OPTUNA_ROOT / "report"
FIGURE_DIR = REPORT_DIR / "figures"
FINAL_VAL_DIR = OPTUNA_ROOT / "final_validation"

STUDY_DB = OPTUNA_ROOT / "optuna_study.db"
TRIALS_CSV = REPORT_DIR / "trials.csv"
IMPORTANCE_CSV = REPORT_DIR / "parameter_importance.csv"
SELECTED_JSON = REPORT_DIR / "selected_trial.json"
FINAL_MULTI_SEED_CSV = REPORT_DIR / "final_multiseed.csv"

for path in [
    OPTUNA_ROOT,
    TUNE_RUNS_DIR,
    FINAL_RUNS_DIR,
    REPORT_DIR,
    FIGURE_DIR,
    FINAL_VAL_DIR,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )

print("ENVIRONMENT  :", ENVIRONMENT)
print("DATA_ZIP     :", DATA_ZIP)
print("EXTRACT_ROOT :", EXTRACT_ROOT)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("OPTUNA_ROOT  :", OPTUNA_ROOT)
print("WORKERS      :", WORKERS)


# ============================================================
# 이전 Optuna Study DB 자동 복구
# ============================================================

# 자동 탐색이 애매한 경우에만 직접 지정하세요.
RESUME_STUDY_DB_OVERRIDE = None

# Kaggle 예:
# RESUME_STUDY_DB_OVERRIDE = Path(
#     "/kaggle/input/previous-optuna-output/01_yolo_optuna_no_aug/optuna_study.db"
# )
#
# 로컬 예:
# RESUME_STUDY_DB_OVERRIDE = Path(
#     r"C:\path\to\optuna_study.db"
# )


def find_resume_study_db():
    """
    이전 Optuna SQLite DB를 찾습니다.

    Kaggle:
    - 이전 01 Notebook Output 또는 백업 Dataset을 Add Input한 경우
      /kaggle/input/**/optuna_study.db 검색

    Local:
    - 현재 출력 위치의 DB가 있으면 그대로 사용
    - 아니면 현재 폴더/부모 폴더의 대표 경로에서 검색
    """
    if RESUME_STUDY_DB_OVERRIDE is not None:
        path = Path(
            RESUME_STUDY_DB_OVERRIDE
        ).expanduser().resolve()

        if not path.exists():
            raise FileNotFoundError(
                f"지정한 optuna_study.db가 없습니다: {path}"
            )

        return path

    # 현재 working/output 경로에 이미 DB가 있으면 가장 우선
    if STUDY_DB.exists():
        return STUDY_DB.resolve()

    candidates = []

    if IS_KAGGLE:
        if KAGGLE_INPUT_ROOT.exists():
            for path in KAGGLE_INPUT_ROOT.rglob(
                "optuna_study.db"
            ):
                candidates.append(
                    path.resolve()
                )

    else:
        cwd = Path.cwd().resolve()

        for root in [
            cwd,
            *cwd.parents,
        ]:
            direct_candidates = [
                root
                / "01_yolo_optuna_no_aug"
                / "optuna_study.db",

                root
                / "01_yolo_optuna_RESUME"
                / "optuna_study.db",

                root
                / "optuna_study.db",
            ]

            for path in direct_candidates:
                if path.exists():
                    candidates.append(
                        path.resolve()
                    )

    if not candidates:
        return None

    # 'optuna' 경로를 우선하고 문자열 순으로 안정적인 선택
    candidates = sorted(
        set(candidates),
        key=lambda path: (
            0
            if "optuna" in str(path).lower()
            else 1,
            str(path),
        ),
    )

    print("\n이전 optuna_study.db 후보:")

    for index, path in enumerate(
        candidates
    ):
        print(
            f"[{index}] {path}"
        )

    return candidates[0]


RESUME_STUDY_DB = (
    find_resume_study_db()
)


if RESUME_STUDY_DB is None:

    print(
        "\n기존 Optuna DB가 없습니다. "
        "새 Study로 시작합니다."
    )

elif (
    RESUME_STUDY_DB.resolve()
    == STUDY_DB.resolve()
):

    print(
        "\n현재 output 폴더의 Optuna DB를 "
        "그대로 이어서 사용합니다."
    )

    print(
        "DB:",
        STUDY_DB,
    )

else:

    # /kaggle/input은 읽기 전용이므로 working으로 복사해야 합니다.
    # 이미 working DB가 있으면 덮어쓰지 않습니다.
    if not STUDY_DB.exists():

        STUDY_DB.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        shutil.copy2(
            RESUME_STUDY_DB,
            STUDY_DB,
        )

        print(
            "\n이전 Optuna Study DB를 "
            "현재 writable output 경로로 복사했습니다."
        )

        print(
            "FROM:",
            RESUME_STUDY_DB,
        )

        print(
            "TO  :",
            STUDY_DB,
        )

    else:

        print(
            "\n현재 output 경로에 이미 DB가 있어 "
            "그 DB를 우선 사용합니다."
        )

        print(
            "DB:",
            STUDY_DB,
        )

# 8. GPU / Device 확인

In [ ]:
if torch.cuda.is_available():
    DEVICE = 0
    print("CUDA GPU:", torch.cuda.get_device_name(0))
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"VRAM: {total_vram:.2f} GB")

elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Apple MPS 사용")

else:
    DEVICE = "cpu"
    print("CPU 사용")

print("DEVICE =", DEVICE)

## GPU 사용 정책: Kaggle T4 x2에서도 GPU 한 장만 사용

현재 실험에서는 `DEVICE = 0`을 유지합니다.

```text
Kaggle T4 x2
GPU 0 → 사용
GPU 1 → 사용 안 함
```

이유는 모델 품질 비교 조건을 최대한 동일하게 유지하기 위해서입니다.

특히 00 Baseline을 GPU 한 장으로 측정했다면:

```text
00 Baseline     = 1 GPU
01 Optuna       = 1 GPU
02 Augmentation = 1 GPU
```

로 맞추는 편이 가장 깔끔합니다.

또한 현재 Optuna는 epoch별 pruning callback을 YOLO trainer에 붙이므로,
`device=[0, 1]` DDP로 즉시 바꾸는 것보다 single-GPU가 안정적입니다.

로컬에서도 GPU가 있다면 첫 번째 GPU 한 장만 사용합니다.

# 9. processed `data.yaml`을 현재 PC 경로에 맞게 보정

1번 전처리 노트북이 만든 `data.yaml`에 절대경로가 들어 있다면
다른 PC에서 실행했을 때 옛 경로가 남을 수 있습니다.

원본을 수정하지 않고 runtime용 YAML을 별도로 생성합니다.

In [ ]:
with open(DATA_YAML, "r", encoding="utf-8") as file:
    dataset_config = yaml.safe_load(file)

if "names" not in dataset_config:
    raise KeyError("data.yaml에 names가 없습니다.")

runtime_config = {
    "path": str(PROCESSED_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "names": dataset_config["names"],
}

RUNTIME_DATA_YAML = REPORT_DIR / "runtime_data.yaml"

with open(RUNTIME_DATA_YAML, "w", encoding="utf-8") as file:
    yaml.safe_dump(
        runtime_config,
        file,
        allow_unicode=True,
        sort_keys=False,
    )

names_raw = runtime_config["names"]

if isinstance(names_raw, dict):
    CLASS_NAMES = {int(k): str(v) for k, v in names_raw.items()}
else:
    CLASS_NAMES = {i: str(v) for i, v in enumerate(names_raw)}

print(RUNTIME_DATA_YAML.read_text(encoding="utf-8"))
print("클래스 수:", len(CLASS_NAMES))

# 10. 이번 Optuna는 이전 augmentation 결과와 독립적으로 실행

In [ ]:
previous_multiseed_df = pd.DataFrame()
previous_experiments_df = pd.DataFrame()

print(
    "이번 Notebook은 augmentation을 OFF한 상태에서 "
    "하이퍼파라미터만 최적화합니다."
)

# 11. Optuna 실험 예산과 고정 조건

## 권장 기본값

- 모델: `yolo26n.pt`
- `imgsz=640`
- `batch=8`
- Optuna trial: 25개
- trial당 최대 40 epoch
- 처음 5개 trial은 sampler가 충분한 정보를 모으도록 pruning 판단의 기반으로 사용
- 각 trial은 10 epoch 이전에는 pruning하지 않음

### 왜 최종 학습보다 trial epoch를 조금 줄이나?

HPO에서는 여러 후보를 비교하는 것이 목적이므로 계산량을 줄일 필요가 있습니다.

단, 지나치게 짧은 trial은
"초반에 빨리 좋아지는 설정"만 선호하게 만들 수 있습니다.

따라서 최종 선택 후에는 반드시 원래 학습 epoch로 다시 학습합니다.

컴퓨팅 여유가 있다면 `TUNE_EPOCHS`를 기존 실험의 50과 같게 올리는 것이 더 엄격합니다.

In [ ]:
MODEL_NAME = "yolo26n.pt"

IMGSZ = 640
BATCH = 8

TARGET_TOTAL_TRIALS = 25
TUNE_EPOCHS = 40

# Optuna pruning과 별개로 YOLO 자체 early stopping은 사실상 끄는 효과를 냅니다.
# trial 간 비교를 더 일정하게 만들기 위함입니다.
TUNE_PATIENCE = TUNE_EPOCHS

SEED = 42

# trial checkpoint는 평가 후 삭제하여 디스크를 절약합니다.
KEEP_TUNE_RUNS = False

# 최종 확인
FINAL_EPOCHS = 50
FINAL_PATIENCE = 12
FINAL_SEEDS = [42, 123, 777]

# 최고 mAP50-95와 이 범위 안에 들어오는 후보는
# Recall까지 고려할 shortlist로 봅니다.
MAP_SHORTLIST_WINDOW = 0.005

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("TARGET_TOTAL_TRIALS:", TARGET_TOTAL_TRIALS)
print("TUNE_EPOCHS :", TUNE_EPOCHS)
print("FINAL_EPOCHS:", FINAL_EPOCHS)

# 11-1. Optuna 전체 trial에서 이미지 증강 OFF

Baseline과 HPO의 차이를 **하이퍼파라미터 하나로만 제한**하기 위해,
모든 Optuna trial과 최종 재학습에서 같은 no-augmentation 설정을 사용합니다.

Ultralytics Detection 기본값에는 mosaic/HSV/flip/scale 등이 들어갈 수 있으므로
명시적으로 0으로 덮어씁니다.

In [ ]:
NO_AUGMENTATION = {
    "hsv_h": 0.0,
    "hsv_s": 0.0,
    "hsv_v": 0.0,
    "degrees": 0.0,
    "translate": 0.0,
    "scale": 0.0,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.0,
    "bgr": 0.0,
    "mosaic": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,
    "copy_paste": 0.0,
    "close_mosaic": 0,
    "erasing": 0.0,
    "auto_augment": None,
    "augmentations": [],
}

NO_AUGMENTATION

# 12. Optuna Search Space

현재 Ultralytics 튜닝 범위를 참고해 너무 극단적이지 않게 설정합니다.

| 파라미터 | 탐색 범위 | 방식 |
|---|---:|---|
| optimizer | AdamW / SGD | categorical |
| lr0 | 1e-5 ~ 1e-2 | log |
| lrf | 0.01 ~ 1.0 | log |
| momentum | 0.70 ~ 0.98 | linear |
| weight_decay | 1e-6 ~ 1e-3 | log |
| warmup_epochs | 0 ~ 5 | linear |
| cos_lr | True / False | categorical |

### 왜 `optimizer="auto"`를 사용하지 않나?

`auto`는 YOLO가 optimizer와 일부 학습 설정을 자동 결정합니다.

그 상태에서 `lr0` 등을 Optuna가 바꿔도
실제로 우리가 의도한 값이 그대로 적용되지 않을 수 있습니다.

따라서 HPO에서는 optimizer를 명시적으로 선택합니다.

In [ ]:
def suggest_core_hyperparameters(trial: optuna.Trial) -> dict:
    return {
        "optimizer": trial.suggest_categorical(
            "optimizer",
            ["AdamW", "SGD"],
        ),

        "lr0": trial.suggest_float(
            "lr0",
            1e-5,
            1e-2,
            log=True,
        ),

        "lrf": trial.suggest_float(
            "lrf",
            0.01,
            1.0,
            log=True,
        ),

        "momentum": trial.suggest_float(
            "momentum",
            0.70,
            0.98,
        ),

        "weight_decay": trial.suggest_float(
            "weight_decay",
            1e-6,
            1e-3,
            log=True,
        ),

        "warmup_epochs": trial.suggest_float(
            "warmup_epochs",
            0.0,
            5.0,
        ),

        "cos_lr": trial.suggest_categorical(
            "cos_lr",
            [False, True],
        ),
    }

# 13. 성능지표 추출 함수

Optuna는 최종적으로 하나의 값을 maximize하지만,
우리는 각 trial의 다른 지표도 모두 보존합니다.

저장:

- Precision
- Recall
- F1
- mAP50
- mAP75
- mAP50-95
- inference ms/image

mAP50-95가 Optuna objective입니다.

In [ ]:
def extract_metrics(metrics) -> dict:
    box = metrics.box

    precision = float(getattr(box, "mp", np.nan))
    recall = float(getattr(box, "mr", np.nan))

    if (
        np.isfinite(precision)
        and np.isfinite(recall)
        and (precision + recall) > 0
    ):
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = np.nan

    speed = getattr(metrics, "speed", {}) or {}

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mAP50": float(box.map50),
        "mAP75": float(box.map75),
        "mAP50_95": float(box.map),
        "inference_ms_per_image": speed.get("inference", np.nan),
    }

# 14. Pruning: 성능이 나쁜 trial을 일찍 중단하기

Optuna의 큰 장점 중 하나입니다.

YOLO는 매 epoch 뒤 Validation을 수행합니다.
Ultralytics의 `on_fit_epoch_end` callback은 Train + Validation이 끝난 시점에 호출되므로
이 시점에는 Validation mAP가 존재합니다.

우리는 매 epoch의 `mAP50-95`를 Optuna에 전달합니다.

Optuna가:

> "현재까지의 다른 trial과 비교했을 때 이 trial은 가능성이 매우 낮다."

라고 판단하면 `trainer.stop=True`로 해당 학습을 종료합니다.

### 주의

초반 몇 epoch만 보고 좋은 모델을 잘못 버리면 안 됩니다.

그래서 아래 Study 설정에서:

- startup trial 수
- warmup epoch

를 둡니다.

In [ ]:
def find_map50_95_in_trainer_metrics(metrics_dict):
    if not metrics_dict:
        return None

    for key, value in metrics_dict.items():
        normalized_key = str(key).lower().replace(" ", "")

        if "map50-95" in normalized_key:
            try:
                return float(value)
            except Exception:
                return None

    return None


def make_optuna_pruning_callback(
    trial: optuna.Trial,
    max_epochs: int,
):
    state = {
        "pruned": False,
        "last_step": -1,
        "last_value": None,
    }

    def callback(trainer):
        step = int(getattr(trainer, "epoch", -1))

        # final evaluation 등에서 같은 step이 다시 호출될 수 있으므로 중복 보고 방지
        if step <= state["last_step"]:
            return

        # 정상 trial 최대 epoch 바깥에서 호출되는 final evaluation 방지
        if step >= max_epochs:
            return

        value = find_map50_95_in_trainer_metrics(
            getattr(trainer, "metrics", {})
        )

        if value is None or not np.isfinite(value):
            return

        state["last_step"] = step
        state["last_value"] = value

        trial.report(
            value,
            step=step,
        )

        if trial.should_prune():
            state["pruned"] = True
            trainer.stop = True

    return callback, state

# 15. trial 하나가 수행하는 전체 과정

한 trial의 순서는 다음과 같습니다.

```text
Optuna가 파라미터 제안
        ↓
새 pretrained YOLO26n 생성
        ↓
B01 default augmentation 조건에서 학습
        ↓
epoch마다 mAP50-95 → Optuna에 report
        ↓
나쁜 trial이면 pruning
        ↓
완료되면 best.pt 다시 로드
        ↓
같은 Validation에서 정식 평가
        ↓
Precision / Recall / mAP 기록
        ↓
trial 임시 checkpoint 정리
```

### 매우 중요

모든 trial은 **새 `YOLO(MODEL_NAME)`에서 시작**합니다.

이전 trial의 weight에서 다음 trial을 이어서 학습하면
공정한 비교가 아닙니다.

In [ ]:
def cleanup_memory():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def objective(trial: optuna.Trial) -> float:
    cleanup_memory()

    params = suggest_core_hyperparameters(trial)

    run_name = f"trial_{trial.number:04d}"

    model = YOLO(MODEL_NAME)

    pruning_callback, pruning_state = make_optuna_pruning_callback(
        trial,
        max_epochs=TUNE_EPOCHS,
    )

    model.add_callback(
        "on_fit_epoch_end",
        pruning_callback,
    )

    started = time.perf_counter()

    try:
        model.train(
            data=str(RUNTIME_DATA_YAML),

            epochs=TUNE_EPOCHS,
            imgsz=IMGSZ,
            batch=BATCH,
            patience=TUNE_PATIENCE,

            device=DEVICE,
            workers=WORKERS,

            seed=SEED,
            deterministic=True,

            # Baseline과 공정하게 비교하기 위해 augmentation은 모두 OFF입니다.

            project=str(TUNE_RUNS_DIR),
            name=run_name,
            exist_ok=True,

            save=True,
            plots=False,
            verbose=False,

            **NO_AUGMENTATION,
            **params,
        )

        train_minutes = (time.perf_counter() - started) / 60.0
        save_dir = Path(model.trainer.save_dir)

        # pruning callback이 stop을 요청했다면 trial 상태를 PRUNED로 남김
        if pruning_state["pruned"]:
            if not KEEP_TUNE_RUNS:
                shutil.rmtree(save_dir, ignore_errors=True)

            raise optuna.TrialPruned(
                f"Pruned around epoch {pruning_state['last_step'] + 1}"
            )

        best_pt = save_dir / "weights" / "best.pt"

        if not best_pt.exists():
            raise FileNotFoundError(best_pt)

        best_model = YOLO(str(best_pt))

        val_metrics = best_model.val(
            data=str(RUNTIME_DATA_YAML),
            split="val",

            imgsz=IMGSZ,
            batch=BATCH,
            device=DEVICE,
            workers=WORKERS,

            plots=False,
            verbose=False,
        )

        metric_dict = extract_metrics(val_metrics)

        # Optuna trial DB에 부가지표 기록
        for key, value in metric_dict.items():
            if value is not None and np.isfinite(value):
                trial.set_user_attr(key, float(value))

        trial.set_user_attr(
            "train_minutes",
            float(train_minutes),
        )

        trial.set_user_attr(
            "actual_epochs",
            int(getattr(model.trainer, "epoch", -1)) + 1,
        )

        objective_value = metric_dict["mAP50_95"]

        if not KEEP_TUNE_RUNS:
            shutil.rmtree(save_dir, ignore_errors=True)

        del best_model
        del model
        cleanup_memory()

        return objective_value

    except optuna.TrialPruned:
        del model
        cleanup_memory()
        raise

    except torch.cuda.OutOfMemoryError:
        cleanup_memory()
        raise optuna.TrialPruned("CUDA OOM")

    except Exception:
        cleanup_memory()
        raise

# 16. Optuna Study 생성

## Sampler: TPESampler

완전히 랜덤하게만 고르는 것이 아니라
지금까지 좋았던 trial 분포를 이용해 다음 값을 제안합니다.

## Pruner: MedianPruner

현재 trial의 중간 성능이
이전에 완료된 trial들의 중간값보다 계속 나쁘다면 pruning 후보가 됩니다.

### SQLite를 사용하는 이유

Study 결과가 메모리에만 있으면 Notebook이 꺼질 때 사라질 수 있습니다.

`optuna_study.db`에 저장하면 같은 Study 이름으로 다시 실행해 이어갈 수 있습니다.

In [ ]:
STUDY_NAME = "yolo26n_core_hpo"

storage_url = f"sqlite:///{STUDY_DB.as_posix()}"

sampler = optuna.samplers.TPESampler(
    seed=SEED,
    multivariate=True,
)

pruner = optuna.pruners.MedianPruner(
    n_startup_trials=5,
    n_warmup_steps=10,
    interval_steps=1,
)

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=storage_url,

    direction="maximize",

    sampler=sampler,
    pruner=pruner,

    load_if_exists=True,
)

print("Study name :", STUDY_NAME)
print("Storage    :", STUDY_DB)
print("Trials now :", len(study.trials))

# 17. Optuna 실행 — 총 25개를 목표로 이어서 실행

이전 버전에서 가장 중요한 수정 부분입니다.

`n_trials=25`는 Study 전체를 25개로 맞추는 옵션이 아니라
**호출할 때마다 최대 25개 trial을 추가**하는 의미입니다.

그래서 기존에 5개 trial이 있었는데 다시:

```python
study.optimize(
    objective,
    n_trials=25,
)
```

를 실행하면 전체가 약 30개가 될 수 있습니다.

이번 통합본은 다음 방식으로 동작합니다.

```python
TARGET_TOTAL_TRIALS = 25
```

그리고 현재 Study에서:

```text
COMPLETE + PRUNED
```

상태의 trial 수를 계산합니다.

예:

```text
COMPLETE/PRUNED = 5
TARGET_TOTAL_TRIALS = 25

남은 목표 = 20
```

따라서 기존 5개는 그대로 살리고 새 trial 20개만 수행합니다.

### RUNNING으로 남은 trial

Kaggle 세션이 강제로 종료되면 마지막 trial이 `RUNNING` 상태로 DB에 남을 수 있습니다.

이 Notebook은 시작할 때 오래된 `RUNNING` trial을 `FAIL`로 정리합니다.

해당 trial의 epoch 중간 checkpoint에서 이어서 학습하지는 않습니다.

In [ ]:
RUN_OPTUNA = True


def summarize_trial_states(
    study,
) -> dict:
    """
    Study에 현재 어떤 상태의 trial이 몇 개 있는지 요약합니다.
    """
    counts = {}

    for trial in study.trials:
        state_name = (
            trial.state.name
        )

        counts[state_name] = (
            counts.get(
                state_name,
                0,
            )
            + 1
        )

    return counts


def mark_stale_running_trials_failed(
    study,
):
    """
    이전 세션 강제 종료 때문에 DB에 RUNNING으로 남아 있는 trial을 FAIL로 정리합니다.

    현재 Notebook은 YOLO epoch 내부 checkpoint resume를 하지 않으므로
    RUNNING trial을 그대로 두지 않습니다.
    """
    running_trials = [
        trial
        for trial in study.trials
        if trial.state
        == TrialState.RUNNING
    ]

    if not running_trials:
        print(
            "stale RUNNING trial 없음"
        )

        return

    print(
        "stale RUNNING trial:",
        [
            trial.number
            for trial
            in running_trials
        ],
    )

    for trial in running_trials:

        try:

            study.tell(
                trial.number,
                state=TrialState.FAIL,
                skip_if_finished=True,
            )

            print(
                f"Trial {trial.number} "
                "-> FAIL 처리"
            )

        except Exception as error:

            print(
                f"Trial {trial.number} "
                "상태 변경 실패:",
                repr(error),
            )


def valid_trial_count(
    study,
) -> int:
    """
    목표 25개에 포함할 유효 trial 개수.

    COMPLETE:
        끝까지 정상 완료

    PRUNED:
        Optuna가 성능이 낮다고 판단해 조기 중단
        → Optuna 탐색에서 정상적인 trial 결과이므로 개수에 포함

    FAIL:
        오류 / 세션 강제 종료
        → 목표 trial 개수에 포함하지 않음
    """
    valid_states = {
        TrialState.COMPLETE,
        TrialState.PRUNED,
    }

    return sum(
        trial.state
        in valid_states
        for trial
        in study.trials
    )


# ------------------------------------------------------------
# 이전 세션의 끊긴 RUNNING trial 정리
# ------------------------------------------------------------

mark_stale_running_trials_failed(
    study
)


print(
    "\n현재 Trial 상태:",
    summarize_trial_states(
        study
    ),
)


already_counted = (
    valid_trial_count(
        study
    )
)


remaining_trials = max(
    0,
    TARGET_TOTAL_TRIALS
    - already_counted,
)


print(
    "\n" + "=" * 80
)

print(
    f"현재 유효 Trial : "
    f"{already_counted}"
)

print(
    f"목표 유효 Trial : "
    f"{TARGET_TOTAL_TRIALS}"
)

print(
    f"남은 Trial 목표 : "
    f"{remaining_trials}"
)

print(
    "=" * 80
)


# ------------------------------------------------------------
# 총 유효 trial이 25개가 될 때까지만 추가
# ------------------------------------------------------------

if RUN_OPTUNA:

    while (
        valid_trial_count(
            study
        )
        < TARGET_TOTAL_TRIALS
    ):

        before_count = (
            valid_trial_count(
                study
            )
        )

        print(
            "\n" + "=" * 90
        )

        print(
            "Optuna 진행상황: "
            f"{before_count}/"
            f"{TARGET_TOTAL_TRIALS}"
        )

        print(
            "=" * 90
        )

        # 중요한 차이:
        # 25개를 한 번에 추가하지 않고 1 trial씩 수행합니다.
        # 따라서 중간에 다시 끊겨도 DB에 저장된 trial들을 세고 이어갈 수 있습니다.
        study.optimize(
            objective,
            n_trials=1,
            n_jobs=1,
            gc_after_trial=True,
            show_progress_bar=False,
        )

        after_count = (
            valid_trial_count(
                study
            )
        )

        print(
            "현재 유효 Trial:",
            f"{after_count}/"
            f"{TARGET_TOTAL_TRIALS}",
        )


else:

    print(
        "RUN_OPTUNA=False"
    )


print(
    "\n최종 Trial 상태:",
    summarize_trial_states(
        study
    ),
)

print(
    "최종 유효 Trial:",
    valid_trial_count(
        study
    ),
    "/",
    TARGET_TOTAL_TRIALS,
)

## 17-1. 현재 Study 전체 Trial 상태 확인

Resume가 제대로 되었는지 확인하려면 아래 표를 봅니다.

특히:

```text
COMPLETE
PRUNED
FAIL
```

개수를 확인하세요.

`RUNNING`이 남아 있으면 정상적인 재개 상태가 아닙니다.

# Study DB 보존 체크

Optuna를 중간에 다시 이어야 할 가능성이 있다면 가장 중요한 파일은:

```text
optuna_study.db
```

입니다.

Kaggle에서는 Notebook 실행 후 반드시 Save Version을 해서
`/kaggle/working` 결과를 보존하세요.

다음 셀은 현재 DB 위치와 크기를 출력합니다.

In [ ]:
if STUDY_DB.exists():

    print(
        "Optuna DB:",
        STUDY_DB,
    )

    print(
        "DB size:",
        f"{STUDY_DB.stat().st_size / 1024:.1f} KB",
    )

else:

    print(
        "아직 optuna_study.db가 생성되지 않았습니다."
    )

In [ ]:
trial_status_df = (
    study.trials_dataframe(
        attrs=(
            "number",
            "state",
            "value",
            "params",
            "datetime_start",
            "datetime_complete",
        )
    )
)

display(
    trial_status_df
)

print(
    trial_status_df[
        "state"
    ].value_counts(
        dropna=False
    )
)

# 18. Best trial 확인

In [ ]:
print("Best trial number:", study.best_trial.number)
print("Best mAP50-95    :", study.best_value)

print("\nBest parameters")
for key, value in study.best_params.items():
    print(f"{key:>20} : {value}")

# 19. 모든 trial을 CSV로 정리

Optuna DB는 재개용으로 좋지만,
사람이 확인하기에는 CSV가 편합니다.

각 trial의:

- state
- objective mAP50-95
- hyperparameter
- Precision
- Recall
- F1
- mAP50
- inference speed
- 학습시간

을 하나의 표로 만듭니다.

In [ ]:
trial_rows = []

for trial in study.trials:
    row = {
        "trial": trial.number,
        "state": trial.state.name,
        "mAP50_95": trial.value,
        **{
            f"param_{key}": value
            for key, value in trial.params.items()
        },
        **{
            key: value
            for key, value in trial.user_attrs.items()
        },
    }

    trial_rows.append(row)

trials_df = pd.DataFrame(trial_rows)

if "mAP50_95" in trials_df.columns:
    trials_df = trials_df.sort_values(
        "mAP50_95",
        ascending=False,
        na_position="last",
    )

trials_df.to_csv(
    TRIALS_CSV,
    index=False,
    encoding="utf-8-sig",
)

display(trials_df)
print("Saved:", TRIALS_CSV)

# 20. Trial이 진행될수록 Best score가 어떻게 변했는지 확인

In [ ]:
completed_df = trials_df[
    trials_df["state"].eq("COMPLETE")
    & trials_df["mAP50_95"].notna()
].copy()

completed_by_trial = completed_df.sort_values("trial")

if len(completed_by_trial):
    completed_by_trial["best_so_far"] = (
        completed_by_trial["mAP50_95"].cummax()
    )

    plt.figure(figsize=(10, 5))
    plt.plot(
        completed_by_trial["trial"],
        completed_by_trial["mAP50_95"],
        marker="o",
        alpha=0.5,
        label="trial mAP50-95",
    )
    plt.plot(
        completed_by_trial["trial"],
        completed_by_trial["best_so_far"],
        linewidth=2,
        label="best so far",
    )
    plt.xlabel("Trial")
    plt.ylabel("Validation mAP50-95")
    plt.title("Optuna Optimization History")
    plt.legend()
    plt.tight_layout()

    figure_path = FIGURE_DIR / "optimization_history.png"
    plt.savefig(figure_path, dpi=160, bbox_inches="tight")
    plt.show()

    print("Saved:", figure_path)

# 21. 파라미터 중요도

Optuna는 trial 결과를 바탕으로:

> "현재 탐색 범위에서 어떤 파라미터가 objective 변화와 가장 관련이 컸는가?"

를 추정할 수 있습니다.

주의:

- 이것은 인과관계를 증명하는 값이 아닙니다.
- trial 수가 너무 적으면 중요도도 흔들립니다.
- 파라미터끼리 상호작용할 수 있습니다.

그래도 다음 탐색 공간을 좁힐 때 유용합니다.

In [ ]:
importance_df = pd.DataFrame()

if len(completed_df) >= 5:
    try:
        importance = optuna.importance.get_param_importances(study)

        importance_df = pd.DataFrame({
            "parameter": list(importance.keys()),
            "importance": list(importance.values()),
        })

        importance_df.to_csv(
            IMPORTANCE_CSV,
            index=False,
            encoding="utf-8-sig",
        )

        display(importance_df)

        plt.figure(figsize=(9, 5))
        plot_df = importance_df.sort_values("importance")
        plt.barh(
            plot_df["parameter"],
            plot_df["importance"],
        )
        plt.xlabel("Estimated importance")
        plt.title("Optuna Hyperparameter Importance")
        plt.tight_layout()

        figure_path = FIGURE_DIR / "parameter_importance.png"
        plt.savefig(figure_path, dpi=160, bbox_inches="tight")
        plt.show()

        print("Saved:", IMPORTANCE_CSV)
        print("Saved:", figure_path)

    except Exception as error:
        print("Parameter importance 계산 생략:", error)
else:
    print("완료 trial이 너무 적어 중요도 계산을 생략합니다.")

# 22. 각 파라미터와 mAP50-95 관계 보기

단순 중요도 숫자만 보지 않고 scatter를 함께 봅니다.

예를 들어 `lr0`가 너무 큰 영역에서 성능이 계속 낮다면
다음 Optuna 실행에서는 search range 자체를 줄일 수 있습니다.

In [ ]:
parameter_columns = [
    column
    for column in completed_df.columns
    if column.startswith("param_")
]

for column in parameter_columns:
    values = completed_df[column]

    # 문자열 categorical 파라미터는 별도 box-style scatter
    if values.dtype == object:
        categories = list(values.dropna().unique())
        category_to_x = {
            category: index
            for index, category in enumerate(categories)
        }

        x = values.map(category_to_x)

        plt.figure(figsize=(8, 5))
        plt.scatter(
            x,
            completed_df["mAP50_95"],
            alpha=0.7,
        )
        plt.xticks(
            range(len(categories)),
            categories,
        )

    else:
        plt.figure(figsize=(8, 5))
        plt.scatter(
            values,
            completed_df["mAP50_95"],
            alpha=0.7,
        )

        if column in {
            "param_lr0",
            "param_lrf",
            "param_weight_decay",
        }:
            plt.xscale("log")

    plt.xlabel(column.replace("param_", ""))
    plt.ylabel("Validation mAP50-95")
    plt.title(f"{column.replace('param_', '')} vs mAP50-95")
    plt.tight_layout()

    figure_path = FIGURE_DIR / f"{column}_vs_map.png"
    plt.savefig(
        figure_path,
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()

# 23. mAP 하나만 보지 않는 shortlist 선택

Optuna objective는 mAP50-95지만,
최종 후보 선택은 한 단계 더 거칩니다.

## 규칙

1. 가장 높은 mAP50-95를 찾음
2. `best - 0.005` 안에 있는 trial을 shortlist
3. 그 안에서 Recall이 가장 높은 trial을 선택

예:

```text
Trial A: mAP 0.250, Recall 0.21
Trial B: mAP 0.248, Recall 0.27
```

두 모델의 mAP 차이는 0.002뿐이라면
우리 서비스에서는 Trial B를 선택할 여지가 충분합니다.

반대로 mAP 차이가 매우 크다면 Recall 하나 때문에 낮은 mAP 모델을 억지로 선택하지 않습니다.

In [ ]:
if not len(completed_df):
    raise RuntimeError("완료된 Optuna trial이 없습니다.")

best_map = completed_df["mAP50_95"].max()

shortlist_df = completed_df[
    completed_df["mAP50_95"]
    >= best_map - MAP_SHORTLIST_WINDOW
].copy()

sort_columns = []

if "recall" in shortlist_df.columns:
    sort_columns.append("recall")

if "f1" in shortlist_df.columns:
    sort_columns.append("f1")

sort_columns.append("mAP50_95")

shortlist_df = shortlist_df.sort_values(
    sort_columns,
    ascending=False,
)

display(
    shortlist_df[
        [
            column
            for column in [
                "trial",
                "mAP50_95",
                "mAP50",
                "precision",
                "recall",
                "f1",
                *parameter_columns,
            ]
            if column in shortlist_df.columns
        ]
    ]
)

SELECTED_TRIAL_NUMBER = int(
    shortlist_df.iloc[0]["trial"]
)

selected_trial = study.trials[SELECTED_TRIAL_NUMBER]

print("Selected trial :", SELECTED_TRIAL_NUMBER)
print("Objective mAP  :", selected_trial.value)
print("Recall         :", selected_trial.user_attrs.get("recall"))
print("Parameters     :", selected_trial.params)

# 24. 선택된 하이퍼파라미터 저장

이 파일은 나중에:

- 최종 학습
- API 모델 재학습
- 팀원 공유
- 실험 보고서

에 사용할 수 있습니다.

In [ ]:
selected_record = {
    "selected_trial": SELECTED_TRIAL_NUMBER,
    "selection_rule": (
        f"mAP50-95 within {MAP_SHORTLIST_WINDOW} of best, "
        "then higher Recall/F1 preferred"
    ),
    "objective_mAP50_95": selected_trial.value,
    "metrics": selected_trial.user_attrs,
    "params": selected_trial.params,
    "fixed": {
        "model": MODEL_NAME,
        "imgsz": IMGSZ,
        "batch": BATCH,
        "tune_epochs": TUNE_EPOCHS,
    },
}

with open(
    SELECTED_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        selected_record,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(json.dumps(selected_record, ensure_ascii=False, indent=2))
print("Saved:", SELECTED_JSON)

# 25. Optuna Best를 여러 seed로 다시 학습

HPO의 Best trial도 한 번만 잘 나온 우연일 수 있습니다.

따라서 최종 파라미터를:

```text
seed 42
seed 123
seed 777
```

로 다시 학습합니다.

이 단계에서는 trial 때보다 더 실제 실험에 가까운 `FINAL_EPOCHS=50`을 사용합니다.

그리고 이 checkpoint들은 삭제하지 않습니다.

실제 최종 모델 후보이기 때문입니다.

In [ ]:
def train_final_seed(
    params: dict,
    seed: int,
    policy_name: str = "B01_tuned",
    data_yaml: Path = RUNTIME_DATA_YAML,
    augmentation_args: dict | None = None,
    project_dir: Path = FINAL_RUNS_DIR,
):
    cleanup_memory()

    model = YOLO(MODEL_NAME)

    train_kwargs = {
        "data": str(data_yaml),
        "epochs": FINAL_EPOCHS,
        "imgsz": IMGSZ,
        "batch": BATCH,
        "patience": FINAL_PATIENCE,

        "device": DEVICE,
        "workers": WORKERS,

        "seed": seed,
        "deterministic": True,

        "project": str(project_dir),
        "name": f"{policy_name}_seed{seed}",
        "exist_ok": True,

        "save": True,
        "plots": True,
        "verbose": True,

        **NO_AUGMENTATION,
        **params,
    }

    if augmentation_args is not None:
        train_kwargs.update(augmentation_args)

    started = time.perf_counter()

    model.train(**train_kwargs)

    train_minutes = (
        time.perf_counter() - started
    ) / 60.0

    save_dir = Path(model.trainer.save_dir)
    best_pt = save_dir / "weights" / "best.pt"

    if not best_pt.exists():
        raise FileNotFoundError(best_pt)

    best_model = YOLO(str(best_pt))

    val_metrics = best_model.val(
        data=str(data_yaml),
        split="val",

        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,

        plots=True,
        project=str(FINAL_VAL_DIR),
        name=f"{policy_name}_seed{seed}",
        exist_ok=True,
        verbose=False,
    )

    row = {
        "policy": policy_name,
        "seed": seed,
        "train_minutes": train_minutes,
        "best_pt": str(best_pt),
        "save_dir": str(save_dir),
        **extract_metrics(val_metrics),
    }

    del best_model
    del model
    cleanup_memory()

    return row

In [ ]:
RUN_FINAL_MULTI_SEED = True

selected_params = dict(selected_trial.params)

final_rows = []

if RUN_FINAL_MULTI_SEED:
    for seed in FINAL_SEEDS:
        print("=" * 90)
        print("Final Optuna Best (no augmentation) - seed", seed)

        final_rows.append(
            train_final_seed(
                params=selected_params,
                seed=seed,
                policy_name="optuna_best_no_aug",
            )
        )

        pd.DataFrame(final_rows).to_csv(
            FINAL_MULTI_SEED_CSV,
            index=False,
            encoding="utf-8-sig",
        )

final_multiseed_df = pd.DataFrame(final_rows)

if FINAL_MULTI_SEED_CSV.exists():
    final_multiseed_df = pd.read_csv(
        FINAL_MULTI_SEED_CSV
    )

display(final_multiseed_df)

# 26. 최종 tuned B01의 평균과 표준편차

In [ ]:
if len(final_multiseed_df):
    tuned_summary = (
        final_multiseed_df
        .groupby("policy")
        .agg(
            runs=("mAP50_95", "count"),
            precision_mean=("precision", "mean"),
            precision_std=("precision", "std"),
            recall_mean=("recall", "mean"),
            recall_std=("recall", "std"),
            f1_mean=("f1", "mean"),
            mAP50_mean=("mAP50", "mean"),
            mAP50_95_mean=("mAP50_95", "mean"),
            mAP50_95_std=("mAP50_95", "std"),
            inference_ms_mean=("inference_ms_per_image", "mean"),
        )
        .reset_index()
    )

    display(tuned_summary)

if len(final_multiseed_df):
    plot_df = final_multiseed_df.sort_values("seed")

    plt.figure(figsize=(8, 5))
    plt.bar(
        plot_df["seed"].astype(str),
        plot_df["mAP50_95"],
    )
    plt.axhline(
        plot_df["mAP50_95"].mean(),
        linestyle="--",
        label=f"mean={plot_df['mAP50_95'].mean():.4f}",
    )
    plt.xlabel("Seed")
    plt.ylabel("Validation mAP50-95")
    plt.title("Optuna Best - Multi-seed mAP50-95")
    plt.legend()
    plt.tight_layout()

    figure_path = FIGURE_DIR / "final_multiseed_map.png"
    plt.savefig(figure_path, dpi=160, bbox_inches="tight")
    plt.show()

    print("Saved:", figure_path)


# 27. 기존 B01과 Optuna tuned B01 비교

이전 augmentation report가 남아 있다면,
기존 B01 multi-seed 평균과 새 tuned B01을 나란히 비교합니다.

여기서 성능이 실제로 향상되어야
Optuna가 "Validation trial 하나에만 과적합"한 것이 아니라
의미 있는 후보를 찾았다고 볼 근거가 더 생깁니다.

In [ ]:
comparison_rows = []

if len(previous_multiseed_df):
    old_b01 = previous_multiseed_df[
        previous_multiseed_df["id"].eq("B01")
    ]

    if len(old_b01):
        row = old_b01.iloc[0]

        comparison_rows.append({
            "setting": "B01_before_optuna",
            "mAP50_95_mean": row["mAP50_95_mean"],
            "mAP50_95_std": row["mAP50_95_std"],
            "precision_mean": row["precision_mean"],
            "recall_mean": row["recall_mean"],
        })

if len(final_multiseed_df):
    comparison_rows.append({
        "setting": "B01_after_optuna",
        "mAP50_95_mean": final_multiseed_df["mAP50_95"].mean(),
        "mAP50_95_std": final_multiseed_df["mAP50_95"].std(),
        "precision_mean": final_multiseed_df["precision"].mean(),
        "recall_mean": final_multiseed_df["recall"].mean(),
    })

optuna_before_after_df = pd.DataFrame(
    comparison_rows
)

display(optuna_before_after_df)

# 28. 이 Notebook은 여기까지가 핵심

이 파일은 **HPO 효과만 분리해서 측정**하기 위한 Notebook입니다.

따라서 여기서 augmentation 조합은 다시 시험하지 않습니다.

다음 단계에서는 별도의 augmentation Notebook에서:

```text
optimizer = auto
augmentation만 변경
```

조건으로 Baseline 대비 augmentation 효과를 독립적으로 비교합니다.

최종 서비스 모델을 만들 때에는 별도 실험에서
Optuna Best + Best Augmentation을 결합할 수 있습니다.

# 29. H04 재현을 위한 OpenCV JPEG static dataset

기존 H04는 단순 YOLO 옵션만 바꾼 것이 아닙니다.

먼저 OpenCV로 JPEG compression을 적용한 static train 데이터를 만들고,
그 위에 YOLO HSV/geometry 증강을 추가했습니다.

이전 실험의 JPEG 정책:

```text
80% 확률로 JPEG compression 적용
quality = 45 ~ 85 사이 랜덤
20%는 원본 그대로
```

Validation은 항상 원본 processed Validation을 사용합니다.

In [ ]:
TRAIN_IMAGE_DIR = PROCESSED_DIR / "images" / "train"
TRAIN_LABEL_DIR = PROCESSED_DIR / "labels" / "train"
VAL_IMAGE_DIR = PROCESSED_DIR / "images" / "val"

train_images = sorted(
    path
    for path in TRAIN_IMAGE_DIR.iterdir()
    if path.is_file()
)


def image_to_label_path(image_path: Path) -> Path:
    return (
        TRAIN_LABEL_DIR
        / f"{image_path.stem}.txt"
    )


def stable_seed(*parts) -> int:
    text = "|".join(map(str, parts))

    return int(
        hashlib.sha1(
            text.encode("utf-8")
        ).hexdigest()[:8],
        16,
    )


def build_h04_jpeg_dataset(
    seed: int,
    apply_probability: float = 0.80,
):
    dataset_dir = (
        AUG_RECHECK_DIR
        / "h04_jpeg_datasets"
        / f"seed_{seed}"
    )

    image_dir = dataset_dir / "images" / "train"
    label_dir = dataset_dir / "labels" / "train"
    yaml_path = dataset_dir / "data.yaml"

    if yaml_path.exists():
        return yaml_path

    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    rows = []

    for source_image in train_images:
        source_label = image_to_label_path(
            source_image
        )

        image = cv2.imread(
            str(source_image)
        )

        if image is None:
            raise ValueError(
                f"이미지 읽기 실패: {source_image}"
            )

        rng = np.random.default_rng(
            stable_seed(
                "H04",
                seed,
                source_image.name,
            )
        )

        apply_aug = (
            rng.random()
            < apply_probability
        )

        output_image = (
            image_dir
            / f"{source_image.stem}.jpg"
        )

        if apply_aug:
            quality = int(
                rng.integers(45, 86)
            )

            success, encoded = cv2.imencode(
                ".jpg",
                image,
                [
                    cv2.IMWRITE_JPEG_QUALITY,
                    quality,
                ],
            )

            if not success:
                raise IOError(
                    f"JPEG encode 실패: {source_image}"
                )

            degraded = cv2.imdecode(
                encoded,
                cv2.IMREAD_COLOR,
            )

            success = cv2.imwrite(
                str(output_image),
                degraded,
                [cv2.IMWRITE_JPEG_QUALITY, 95],
            )

            if not success:
                raise IOError(
                    f"JPEG 저장 실패: {output_image}"
                )

        else:
            # 확장자를 통일해야 하는 경우 decode/rewrite가 아니라
            # 기존 이미지가 jpg라는 현재 processed 구조를 전제로 bytes copy
            shutil.copy2(
                source_image,
                output_image,
            )

        shutil.copy2(
            source_label,
            label_dir / source_label.name,
        )

        rows.append({
            "source_image": source_image.name,
            "output_image": output_image.name,
            "augmented": apply_aug,
        })

    pd.DataFrame(rows).to_csv(
        dataset_dir / "generation_report.csv",
        index=False,
        encoding="utf-8-sig",
    )

    config = {
        "train": str(image_dir.resolve()),
        "val": str(VAL_IMAGE_DIR.resolve()),
        "names": {
            class_id: name
            for class_id, name
            in CLASS_NAMES.items()
        },
    }

    with open(
        yaml_path,
        "w",
        encoding="utf-8",
    ) as file:
        yaml.safe_dump(
            config,
            file,
            allow_unicode=True,
            sort_keys=False,
        )

    return yaml_path

# 30. H04 / Y14 augmentation 설정

이 값들은 앞 augmentation 실험에서 사용한 설정을 그대로 재현합니다.

## H04

```text
OpenCV JPEG compression
+
YOLO HSV
+
rotation / translation / scale / horizontal flip
```

## Y14

```text
Mosaic 0.65
MixUp 0.10
CutMix 0.10
```

여기에서는 **학습 하이퍼파라미터는 Optuna Best로 고정**합니다.

In [ ]:
NO_AUG = {
    "hsv_h": 0.0,
    "hsv_s": 0.0,
    "hsv_v": 0.0,

    "degrees": 0.0,
    "translate": 0.0,
    "scale": 0.0,
    "shear": 0.0,
    "perspective": 0.0,

    "flipud": 0.0,
    "fliplr": 0.0,

    "mosaic": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,
}


H04_AUG = {
    **NO_AUG,

    "hsv_h": 0.012,
    "hsv_s": 0.40,
    "hsv_v": 0.30,

    "degrees": 7.0,
    "translate": 0.05,
    "scale": 0.15,

    "fliplr": 0.40,
}


Y14_AUG = {
    **NO_AUG,

    "mosaic": 0.65,
    "mixup": 0.10,
    "cutmix": 0.10,
    "close_mosaic": 10,
}


def supported_args(config: dict) -> dict:
    if not DEFAULT_CFG_DICT:
        return config

    supported = set(DEFAULT_CFG_DICT.keys())

    unknown = [
        key
        for key in config
        if key not in supported
    ]

    if unknown:
        print(
            "현재 Ultralytics에서 지원되지 않아 제외:",
            unknown,
        )

    return {
        key: value
        for key, value in config.items()
        if key in supported
    }

# 31. Tuned B01 / H04 / Y14 재검증

이 단계는 Optuna가 끝난 뒤 수행합니다.

### B01
- processed 원본
- YOLO default augmentation

### H04
- JPEG static train
- H04 YOLO 증강

### Y14
- processed 원본
- Mosaic/MixUp/CutMix

세 후보를 같은:

- optimized training parameters
- epochs
- imgsz
- batch
- seeds

로 비교합니다.

In [ ]:
RUN_AUGMENTATION_RECHECK = False
augmentation_recheck_df = pd.DataFrame()

print("augmentation 재검증은 이 HPO Notebook에서 실행하지 않습니다.")

# 32. B01 / H04 / Y14 재검증 평균 비교

In [ ]:
if len(augmentation_recheck_df):

    recheck_summary_df = (
        augmentation_recheck_df
        .groupby("policy")
        .agg(
            runs=("mAP50_95", "count"),

            mAP50_95_mean=(
                "mAP50_95",
                "mean",
            ),
            mAP50_95_std=(
                "mAP50_95",
                "std",
            ),

            precision_mean=(
                "precision",
                "mean",
            ),

            recall_mean=(
                "recall",
                "mean",
            ),

            f1_mean=(
                "f1",
                "mean",
            ),

            inference_ms_mean=(
                "inference_ms_per_image",
                "mean",
            ),
        )
        .reset_index()
        .sort_values(
            [
                "mAP50_95_mean",
                "recall_mean",
            ],
            ascending=False,
        )
    )

    display(recheck_summary_df)

    plt.figure(figsize=(9, 5))

    plot_df = recheck_summary_df.sort_values(
        "mAP50_95_mean"
    )

    plt.barh(
        plot_df["policy"],
        plot_df["mAP50_95_mean"],
        xerr=plot_df[
            "mAP50_95_std"
        ].fillna(0),
        capsize=4,
    )

    plt.xlabel(
        "Mean Validation mAP50-95"
    )

    plt.title(
        "Tuned Hyperparameters: B01 vs H04 vs Y14"
    )

    plt.tight_layout()

    figure_path = (
        FIGURE_DIR
        / "augmentation_recheck_map.png"
    )

    plt.savefig(
        figure_path,
        dpi=160,
        bbox_inches="tight",
    )

    plt.show()

# 33. Optuna 결과를 볼 때의 기준

1. `mAP50-95` multi-seed 평균
2. Recall
3. seed별 표준편차
4. Precision / F1
5. 최종 Validation PR Curve와 Confusion Matrix

특히 한 trial의 최고값만 보고 확정하지 않고,
Best hyperparameters를 여러 seed로 다시 학습한 결과를 확인합니다.

# 34. 독립 Test set이 있다면 마지막에만 평가

현재 augmentation/HPO는 Validation을 반복해서 보고 모델을 선택합니다.

따라서 Validation은 더 이상 완전히 독립적인 최종 시험지가 아닙니다.

`data.yaml`에 나중에 `test:` split을 추가했다면
최종 설정 하나를 고른 뒤 마지막에만 아래 코드를 실행하세요.

Test 결과를 보고 다시 Optuna를 돌리면
그 Test도 사실상 Validation처럼 사용한 것이 됩니다.

In [ ]:
RUN_FINAL_TEST = False

if RUN_FINAL_TEST:

    if "test" not in dataset_config:
        raise KeyError(
            "data.yaml에 test split이 없습니다."
        )

    # 최종적으로 선택한 checkpoint 경로를 직접 지정
    FINAL_MODEL_PT = Path(
        "여기에_최종_best.pt_경로"
    )

    if not FINAL_MODEL_PT.exists():
        raise FileNotFoundError(
            FINAL_MODEL_PT
        )

    final_model = YOLO(
        str(FINAL_MODEL_PT)
    )

    test_metrics = final_model.val(
        data=str(RUNTIME_DATA_YAML),
        split="test",

        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,

        plots=True,

        project=str(
            OPTUNA_ROOT / "final_test"
        ),
        name="test",
        exist_ok=True,
    )

    display(
        pd.DataFrame(
            [extract_metrics(test_metrics)]
        )
    )

# 35. Optuna를 한 번 돌리고 끝내지 말아야 하는 이유

HPO는 보통 한 번의 거대한 탐색보다 **coarse → narrow** 방식이 효율적입니다.

## 1차

지금 Notebook의 넓은 범위:

```text
lr0        1e-5 ~ 1e-2
lrf        0.01 ~ 1.0
momentum   0.70 ~ 0.98
...
```

으로 어디가 좋은지 찾습니다.

## 2차

예를 들어 좋은 trial들이 모두:

```text
lr0 = 0.0007 ~ 0.0015
weight_decay = 0.0002 ~ 0.0005
```

근처에 모였다면,

다음 Study에서는 그 주변만 더 촘촘하게 탐색합니다.

### 추천

1차 20~30 trial  
→ 결과 확인  
→ 중요한 파라미터 범위 축소  
→ 2차 local Optuna  
→ multi-seed

이 방식이 수백 개 trial을 아무 생각 없이 한 번에 돌리는 것보다
실험을 이해하기 쉽습니다.

# 36. Loss gain까지 튜닝하고 싶다면

`box`, `cls`, `dfl`도 하이퍼파라미터입니다.

하지만 현재 데이터는 클래스가 많고 Validation 이미지가 적기 때문에
처음부터 너무 많은 차원을 검색하면 Validation noise를 따라갈 위험이 있습니다.

따라서 다음 순서를 권장합니다.

```text
core HPO
    ↓
lr / optimizer / regularization 범위 확정
    ↓
augmentation 재확인
    ↓
필요하면 box / cls / dfl을 좁은 범위에서 추가 HPO
```

특히 `cls`를 올렸더니 Validation mAP가 우연히 좋아졌다고 해서
바로 최종값으로 확정하지 말고 반드시 multi-seed로 확인하세요.

# 37. 최종 체크리스트

- [ ] `data.zip`에서 Train / Validation이 정상적으로 읽혔다.
- [ ] Baseline과 동일한 YOLO26n / imgsz / batch를 사용했다.
- [ ] Optuna trial 동안 augmentation이 OFF였다.
- [ ] Best trial의 Precision / Recall / F1 / mAP를 확인했다.
- [ ] Parameter importance와 optimization history를 확인했다.
- [ ] Best params를 여러 seed로 다시 학습했다.
- [ ] `final_validation/`의 PR Curve / F1 Curve / Confusion Matrix를 확인했다.
- [ ] `report/selected_trial.json`을 보관했다.
- [ ] `report/final_multiseed.csv`를 보관했다.

이 결과를 `00 Baseline`과 비교하면 하이퍼파라미터 최적화의 효과를 볼 수 있습니다.

---

## Resume 관련 추가 체크

- [ ] `optuna_study.db`를 보존했다.
- [ ] Notebook 재실행 시 기존 DB가 자동 발견되는 것을 확인했다.
- [ ] stale `RUNNING` trial이 `FAIL`로 정리되었다.
- [ ] `COMPLETE + PRUNED` 기준 총 25개까지만 실행되었다.
- [ ] 재실행했다고 trial 25개가 추가로 생기지 않았다.
- [ ] Kaggle T4 x2에서도 이번 실험은 `DEVICE=0` 한 장만 사용했다.

# 참고 문서

- Ultralytics Hyperparameter Tuning  
  https://docs.ultralytics.com/guides/hyperparameter-tuning/

- Ultralytics Train  
  https://docs.ultralytics.com/modes/train/

- Ultralytics Callback  
  https://docs.ultralytics.com/usage/callbacks/

- Optuna Study  
  https://optuna.readthedocs.io/en/stable/reference/study.html

- Optuna Trial / Pruning  
  https://optuna.readthedocs.io/en/stable/reference/trial.html